## Installs

This is the self-contained **demo / inference notebook** for **Group 35 Solution C**.

It implements the final selected Category C system:

- a 3-model raw-text RoBERTa ensemble
- bucketed by total length and length gap
- with a **Long-Text Specialist** applied only to long and extra-long pairs

This notebook expects:

- one input the test CSV file in `/content/data`
- four trained inference bundles in `/content/bundles`


In [1]:
%pip install -q transformers sentencepiece

## Imports


In [2]:
import csv
import json
import re
import shutil
import tempfile
import zipfile
from pathlib import Path

import torch
from torch.utils.data import DataLoader, Dataset
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from google.colab import files

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import roc_auc_score, log_loss, brier_score_loss, confusion_matrix, roc_curve

if torch.cuda.is_available():
    print('device:', torch.cuda.get_device_name(0))
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

device: NVIDIA A100-SXM4-80GB


## Constants


In [ ]:
# ── Paths ──
WORKDIR    = Path('/content/group35_solution_c_demo')
DATA_DIR   = Path('/content/data') # Upload test CSV here
MODEL_DIR  = WORKDIR / 'models' # Extracted model bundles go here
OUTPUT_DIR = WORKDIR / 'outputs' # Final predictions saved here
MODELS_ZIP_PATH = Path('/content/models.zip')

for directory in [WORKDIR, DATA_DIR, MODEL_DIR, OUTPUT_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

# ── Configuration ──
INPUT_CSV_NAME        = 'dev.csv'
FINAL_SUBMISSION_NAME = 'Group_35_C.csv'
BATCH_SIZE            = 8

# The four model bundle directory names
MODEL_NAMES = [
    'transformer_roberta_20epochs',
    'transformer_roberta_hard_negative_precision',
    'transformer_roberta_symmetry',
    'transformer_roberta_long_text',
]

##Config

In [ ]:
# ── Ensemble configuration ──
# Bucket weights and thresholds were optimised on dev.csv via simplex grid search.
# Each of the 12 buckets (4 primary × 3 secondary) has its own weight vector
# over the 3 base models and its own classification threshold.

BASE_ENSEMBLE_CONFIG = {
    'models': [
        'transformer_roberta_20epochs',
        'transformer_roberta_hard_negative_precision',
        'transformer_roberta_symmetry',
    ],
    'primary_buckets': [
        {'name': 'short',  'max_total_words': 120},
        {'name': 'medium', 'max_total_words': 200},
        {'name': 'long',   'max_total_words': 300},
        {'name': 'xlong',  'max_total_words': None},
    ],
    'secondary_buckets': [
        {'name': 'balanced', 'max_abs_len_diff': 20},
        {'name': 'diff',     'max_abs_len_diff': 60},
        {'name': 'verydiff', 'max_abs_len_diff': None},
    ],
    'bucket_configs': {
        'short_balanced':   {'threshold': 0.003337, 'weights': {'transformer_roberta_20epochs': 0.02,  'transformer_roberta_hard_negative_precision': 0.84, 'transformer_roberta_symmetry': 0.14}},
        'short_diff':       {'threshold': 0.002496, 'weights': {'transformer_roberta_20epochs': 0.24,  'transformer_roberta_hard_negative_precision': 0.28, 'transformer_roberta_symmetry': 0.48}},
        'short_verydiff':   {'threshold': 3.6e-05,  'weights': {'transformer_roberta_20epochs': 0.0,   'transformer_roberta_hard_negative_precision': 0.0,  'transformer_roberta_symmetry': 1.0}},
        'medium_balanced':  {'threshold': 0.000509, 'weights': {'transformer_roberta_20epochs': 0.0,   'transformer_roberta_hard_negative_precision': 0.02, 'transformer_roberta_symmetry': 0.98}},
        'medium_diff':      {'threshold': 0.027093, 'weights': {'transformer_roberta_20epochs': 0.04,  'transformer_roberta_hard_negative_precision': 0.96, 'transformer_roberta_symmetry': 0.0}},
        'medium_verydiff':  {'threshold': 0.002196, 'weights': {'transformer_roberta_20epochs': 0.0,   'transformer_roberta_hard_negative_precision': 0.14, 'transformer_roberta_symmetry': 0.86}},
        'long_balanced':    {'threshold': 0.000674, 'weights': {'transformer_roberta_20epochs': 0.02,  'transformer_roberta_hard_negative_precision': 0.1,  'transformer_roberta_symmetry': 0.88}},
        'long_diff':        {'threshold': 0.004013, 'weights': {'transformer_roberta_20epochs': 0.64,  'transformer_roberta_hard_negative_precision': 0.36, 'transformer_roberta_symmetry': 0.0}},
        'long_verydiff':    {'threshold': 0.07847,  'weights': {'transformer_roberta_20epochs': 0.12,  'transformer_roberta_hard_negative_precision': 0.1,  'transformer_roberta_symmetry': 0.78}},
        'xlong_balanced':   {'threshold': 0.010128, 'weights': {'transformer_roberta_20epochs': 0.02,  'transformer_roberta_hard_negative_precision': 0.3,  'transformer_roberta_symmetry': 0.68}},
        'xlong_diff':       {'threshold': 0.000641, 'weights': {'transformer_roberta_20epochs': 0.0,   'transformer_roberta_hard_negative_precision': 0.02, 'transformer_roberta_symmetry': 0.98}},
        'xlong_verydiff':   {'threshold': 0.147718, 'weights': {'transformer_roberta_20epochs': 0.86,  'transformer_roberta_hard_negative_precision': 0.0,  'transformer_roberta_symmetry': 0.14}},
    },
}

# ── Long/xlong specialist repair configuration ──
# For long_* and xlong_* buckets, the base ensemble probability is blended with the
# specialist's probability: P_final = (1-α) * P_base + α * P_specialist.
# α and threshold were optimised per-bucket on dev.csv.
LONG_SPECIALIST_REPAIR = {
    'long_balanced':  {'alpha': 0.01,  'threshold': 0.001852},
    'long_diff':      {'alpha': 0.89,  'threshold': 0.05499},
    'long_verydiff':  {'alpha': 0.12,  'threshold': 0.075792},
    'xlong_balanced': {'alpha': 0.8,   'threshold': 0.268476},
    'xlong_diff':     {'alpha': 0.02,  'threshold': 0.005955},
    'xlong_verydiff': {'alpha': 0.69,  'threshold': 0.072027},
}

## Data Loader and Processing Functions



In [ ]:
def choose_primary_bucket(total_words: int) -> str:
    """Assign a pair to a primary length bucket based on total word count."""

    for bucket in BASE_ENSEMBLE_CONFIG['primary_buckets']:
        if bucket['max_total_words'] is None or total_words <= bucket['max_total_words']:
            return str(bucket['name'])
    return 'xlong'


def choose_secondary_bucket(abs_len_diff: int) -> str:
    """Assign a pair to a secondary bucket based on absolute per-text length difference."""

    for bucket in BASE_ENSEMBLE_CONFIG['secondary_buckets']:
        if bucket['max_abs_len_diff'] is None or abs_len_diff <= bucket['max_abs_len_diff']:
            return str(bucket['name'])
    return 'verydiff'


def load_input_csv(path) -> list:
    """Load a test/dev CSV and compute bucket assignments for each pair.
    
    Returns a list of dicts, each containing raw texts, word counts, bucket keys,
    and optionally labels (None if not present in the CSV).
    """
    
    rows = []
    with Path(path).open('r', encoding='utf-8-sig', newline='') as handle:
        reader = csv.DictReader(handle)
        for row in reader:
            text_1       = str(row['text_1'])
            text_2       = str(row['text_2'])
            len_1        = len(text_1.split())
            len_2        = len(text_2.split())
            total_words  = len_1 + len_2
            abs_len_diff = abs(len_1 - len_2)
            primary_bucket   = choose_primary_bucket(total_words)
            secondary_bucket = choose_secondary_bucket(abs_len_diff)
            rows.append({
                'text_1':            text_1,
                'text_2':            text_2,
                'label':             None if row.get('label', '') == '' else int(float(row['label'])),
                'len_1':             len_1,
                'len_2':             len_2,
                'total_words':       total_words,
                'abs_len_diff':      abs_len_diff,
                'primary_bucket':    primary_bucket,
                'secondary_bucket':  secondary_bucket,
                'bucket_key':        f'{primary_bucket}_{secondary_bucket}',
            })
    return rows

## Text Processing and DataLoader


In [ ]:
def preprocess_text(text, lowercase=False, normalize_urls=False, normalize_emails=False, normalize_numbers=False) -> str:
    """Apply optional text normalisation. All flags are False in the final system (raw text)."""

    processed = str(text)
    if lowercase:
        processed = processed.lower()
    if normalize_urls:
        processed = re.sub(r'(https?://\S+|www\.\S+|urllink)', ' <url> ', processed, flags=re.IGNORECASE)
    if normalize_emails:
        processed = re.sub(r'\b[\w.+-]+@[\w-]+\.[\w.-]+\b', ' <email> ', processed)
    if normalize_numbers:
        processed = re.sub(r'\b\d+(?:[.,:/-]\d+)*\b', ' <num> ', processed)
    return processed


class TransformerPairDataset(Dataset):
    """Inference-time dataset that tokenises text pairs for a single RoBERTa model.
    
    Applies the model's saved preprocessing config and optionally encodes
    the reversed pair for symmetric inference.
    """

    def __init__(self, rows, tokenizer, config, include_reverse_pair=False):
        self.rows                = rows
        self.tokenizer           = tokenizer
        self.config              = config
        self.max_length          = config['max_length']
        self.include_reverse_pair = include_reverse_pair

    def __len__(self):
        return len(self.rows)

    def _encode(self, text_1, text_2) -> dict:
        encoded = self.tokenizer(text_1, text_2, truncation=True, max_length=self.max_length, padding='max_length')
        return {key: torch.tensor(value, dtype=torch.long) for key, value in encoded.items()}

    def __getitem__(self, index) -> dict:
        row  = self.rows[index]
        cfg  = self.config
        text_1 = preprocess_text(row['text_1'], lowercase=cfg.get('lowercase', False), normalize_urls=cfg.get('normalize_urls', False), normalize_emails=cfg.get('normalize_emails', False), normalize_numbers=cfg.get('normalize_numbers', False))
        text_2 = preprocess_text(row['text_2'], lowercase=cfg.get('lowercase', False), normalize_urls=cfg.get('normalize_urls', False), normalize_emails=cfg.get('normalize_emails', False), normalize_numbers=cfg.get('normalize_numbers', False))
        item = self._encode(text_1, text_2)
        if self.include_reverse_pair:
            reverse_item = self._encode(text_2, text_1)
            for key, value in reverse_item.items():
                item[f'reverse_{key}'] = value
        return item


def move_batch_to_device(batch: dict, device: torch.device) -> dict:
    return {key: value.to(device) for key, value in batch.items()}


def get_model_inputs(batch: dict, reverse: bool = False) -> dict:
    prefix = 'reverse_' if reverse else ''
    inputs = {}
    for key, value in batch.items():
        if reverse:
            if key.startswith(prefix):
                inputs[key[len(prefix):]] = value
        elif not key.startswith('reverse_'):
            inputs[key] = value
    return inputs


def probabilities_from_logits(logits: torch.Tensor) -> torch.Tensor:
    return torch.softmax(logits, dim=-1)[:, 1]


def compute_metrics(predictions: list, labels: list) -> dict:
    """Compute accuracy, precision, recall, and F1 from binary predictions and labels."""

    tp = sum(1 for p, y in zip(predictions, labels) if p == 1 and y == 1)
    tn = sum(1 for p, y in zip(predictions, labels) if p == 0 and y == 0)
    fp = sum(1 for p, y in zip(predictions, labels) if p == 1 and y == 0)
    fn = sum(1 for p, y in zip(predictions, labels) if p == 0 and y == 1)
    precision = tp / max(tp + fp, 1)
    recall    = tp / max(tp + fn, 1)
    f1 = 0.0 if precision + recall == 0 else 2 * precision * recall / (precision + recall)
    return {
        'accuracy': (tp + tn) / max(tp + tn + fp + fn, 1),
        'precision': precision,
        'recall':    recall,
        'f1':f1,
        'tp': tp, 'tn': tn, 'fp': fp, 'fn': fn,
    }

## Model Loading and Inference


In [ ]:
def load_model_bundle(model_dir, device) -> dict:
    """Load a model inference bundle from disk.
    
    Reads inference_metadata.json for config and threshold, then loads the
    HuggingFace tokenizer and model weights from the hf_model/ subdirectory.
    """

    metadata  = json.loads((model_dir / 'inference_metadata.json').read_text(encoding='utf-8'))
    tokenizer = AutoTokenizer.from_pretrained(str(model_dir / 'hf_model'))
    model     = AutoModelForSequenceClassification.from_pretrained(str(model_dir / 'hf_model')).to(device)
    return {
        'config':    metadata['config'],
        'threshold': float(metadata.get('threshold', 0.5)),
        'tokenizer': tokenizer,
        'model':     model,
    }


@torch.no_grad()
def predict_probabilities_for_model(bundle, rows, batch_size, device) -> list:
    """Run inference for a single model on all input rows.
    
    If symmetric_inference is enabled in the model's config, averages predictions
    from both pair orderings: P = 0.5 * (P(t1,t2) + P(t2,t1)).
    
    Returns:
        List of float probabilities, one per row.
    """
    
    config              = bundle['config']
    symmetric_inference = bool(config.get('symmetric_inference', False))
    dataset    = TransformerPairDataset(rows, bundle['tokenizer'], config, include_reverse_pair=symmetric_inference)
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=0, pin_memory=device.type == 'cuda')
    probabilities = []
    bundle['model'].eval()
    for batch in dataloader:
        batch   = move_batch_to_device(batch, device)
        outputs = bundle['model'](**get_model_inputs(batch))
        probs   = probabilities_from_logits(outputs.logits)
        if symmetric_inference:
            reverse_outputs = bundle['model'](**get_model_inputs(batch, reverse=True))
            reverse_probs   = probabilities_from_logits(reverse_outputs.logits)
            probs           = 0.5 * (probs + reverse_probs)
        probabilities.extend(probs.detach().cpu().tolist())
    return probabilities

## Inference Execution Block


In [ ]:
# ── Extract model bundles if provided as a single zip ──
if MODELS_ZIP_PATH.exists():
    print(f'Extracting {MODELS_ZIP_PATH} to {MODEL_DIR}...')
    with zipfile.ZipFile(MODELS_ZIP_PATH, 'r') as archive:
        archive.extractall(MODEL_DIR)
    print('Extraction complete.')
else:
    print(f'{MODELS_ZIP_PATH} not found. Assuming models are already extracted in {MODEL_DIR}.')

# ── Load input data and compute bucket assignments ──
rows = load_input_csv(DATA_DIR / INPUT_CSV_NAME)
print('Loaded rows:', len(rows))

# ── Run inference for each of the 4 models ──
single_model_probabilities = {}
for model_name in MODEL_NAMES:
    bundle = load_model_bundle(MODEL_DIR / model_name, device)
    single_model_probabilities[model_name] = predict_probabilities_for_model(bundle, rows, BATCH_SIZE, device)
    print(f'Inferred: {model_name}')

Loaded rows: 5993


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Inferred: transformer_roberta_20epochs


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Inferred: transformer_roberta_hard_negative_precision


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Inferred: transformer_roberta_symmetry


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Inferred: transformer_roberta_long_text


## Ensemble and Predictions

In [ ]:
predictions   = []
detailed_rows = []
labels        = [row['label'] for row in rows] if rows[0]['label'] is not None else None

for index, row in enumerate(rows):
    bucket_key    = row['bucket_key']
    bucket_config = BASE_ENSEMBLE_CONFIG['bucket_configs'][bucket_key]

    # ── Layer 1: Bucketed weighted average of the 3 base models ──
    weighted_sum  = sum(float(w) * single_model_probabilities[m][index] for m, w in bucket_config['weights'].items())
    weight_total  = sum(float(w) for w in bucket_config['weights'].values())
    base_probability = weighted_sum / max(weight_total, 1e-12)

    # ── Layer 2: Specialist repair for long/xlong buckets ──
    specialist_probability = single_model_probabilities['transformer_roberta_long_text'][index]
    final_probability = base_probability
    final_threshold   = float(bucket_config['threshold'])
    specialist_alpha  = 0.0

    if bucket_key in LONG_SPECIALIST_REPAIR:
        repair_cfg        = LONG_SPECIALIST_REPAIR[bucket_key]
        specialist_alpha  = float(repair_cfg['alpha'])
        final_probability = (1.0 - specialist_alpha) * base_probability + specialist_alpha * specialist_probability
        final_threshold   = float(repair_cfg['threshold'])

    # ── Apply bucket-specific threshold to produce binary prediction ──
    prediction = 1 if final_probability >= final_threshold else 0
    predictions.append(prediction)
    detailed_rows.append({
        'index':                index,
        'primary_bucket':       row['primary_bucket'],
        'secondary_bucket':     row['secondary_bucket'],
        'bucket_key':           bucket_key,
        'len_1':                row['len_1'],
        'len_2':                row['len_2'],
        'total_words':          row['total_words'],
        'abs_len_diff':         row['abs_len_diff'],
        'base_probability':     f'{base_probability:.6f}',
        'specialist_probability': f'{specialist_probability:.6f}',
        'specialist_alpha':     f'{specialist_alpha:.2f}',
        'final_probability':    f'{final_probability:.6f}',
        'final_threshold':      f'{final_threshold:.6f}',
        'prediction':           prediction,
        **(({'label': labels[index]}) if labels is not None else {}),
    })

submission_path = OUTPUT_DIR / FINAL_SUBMISSION_NAME
with submission_path.open('w', encoding='utf-8', newline='') as handle:
    writer = csv.DictWriter(handle, fieldnames=['prediction'])
    writer.writeheader()
    for prediction in predictions:
        writer.writerow({'prediction': prediction})

print('Submission:', submission_path)
if labels is not None:
    print(compute_metrics(predictions, labels))

files.download(str(submission_path))

Submission: /content/group35_solution_c_demo/outputs/Group_35_C.csv
{'accuracy': 0.8298014350075088, 'precision': 0.7861157953906689, 'recall': 0.9152486910994765, 'f1': 0.8457816752343513, 'tp': 2797, 'tn': 2176, 'fp': 761, 'fn': 259}


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Extra Evaluation Metrics

In [ ]:
if labels is not None:
    print('--- Additional Evaluation Metrics ---')

    # Extract probabilities from detailed_rows
    final_probs = [float(row['final_probability']) for row in detailed_rows]

    # ROC AUC
    roc_auc = roc_auc_score(labels, final_probs)
    print(f'ROC AUC: {roc_auc:.4f}')

    # Equal Error Rate (EER)
    fpr, tpr, thresholds = roc_curve(labels, final_probs)
    fnr = 1 - tpr
    eer_index = np.nanargmin(np.absolute((fnr - fpr)))
    eer = fpr[eer_index]
    print(f'Equal Error Rate (EER): {eer:.4f}')

    # Log Loss
    ll = log_loss(labels, final_probs)
    print(f'Log Loss: {ll:.4f}')

    # Brier Score
    brier = brier_score_loss(labels, final_probs)
    print(f'Brier Score: {brier:.4f}')
    print('\n--- Confusion Matrix ---')

    # Confusion Matrix Visualization
    cm = confusion_matrix(labels, predictions)
    plt.figure(figsize=(6, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
                xticklabels=['Predicted 0', 'Predicted 1'],
                yticklabels=['Actual 0', 'Actual 1'])
    plt.title('Confusion Matrix')
    plt.xlabel('Predicted Label')
    plt.ylabel('True Label')
    plt.show()
else:
    print('Labels are not available in the input data. Skipping additional evaluation metrics.')